# Session 3: Generative Models, Likelihood and Prediction
### Student Laboratory Workbook
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/03_generative_models_likelihood_prediction.ipynb)

---

## 1. Before Class & Workflow Architecture
* **Core Philosophy**: A Bayesian statistical model is not a passive curve-fitting recipe; it is an active **generative hypothesis** about how observable empirical phenomena are produced in nature or cognition.
* **Workbook Structure**:
  1. **Worked Example**: Each section presents a complete, working simulation with substantive cognitive or social context.
  2. **💡 Suggested Experiments**: Specific ideas, parameter variations, and research questions for you to test.
  3. **🧪 Student Sandbox**: Runnable code blocks ready for your modifications.
* **Runtime**: ~90 minutes. Click **File $\to$ Save a copy in Drive** to preserve your code and experiment notes.


## 2. Environment Check & Setup
Initialize standard scientific computing libraries and configure Plotly for Google Colab.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.integrate import trapezoid
import plotly.graph_objects as go
from plotly.subplots import make_subplots

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print(f"✅ Environment initialized. NumPy random seed set to {RANDOM_SEED}.")


## 3. Observable Learning Targets
By completing this workbook, you will be able to:
1. **Simulate Forward**: Translate verbal data stories into `numpy.random` code that simulates realistic synthetic datasets from known parameter truths.
2. **Evaluate Likelihood Compatibility**: Compute and plot $\mathcal{L}(\theta \mid y)$ by holding observed empirical data fixed while varying candidate models $\theta$.
3. **Quantify Evidence via Likelihood Ratios**: Calculate $\text{LR} = \mathcal{L}(\theta_1) / \mathcal{L}(\theta_2)$ to measure relative compatibility between competing substantive hypotheses.
4. **Distinguish the Bayesian Quartet**: Formally contrast **Prior** $p(\theta)$, **Prior Predictive** $p(y_{\text{sim}})$, **Posterior** $p(\theta \mid y)$, and **Posterior Predictive** $p(\tilde{y} \mid y)$.
5. **Decouple Mean Expectation from Individual Realization**: Differentiate the uncertainty in estimating an average (epistemic) from predicting an individual observable future trial (epistemic + aleatory).
6. **Diagnose Overdispersion**: Identify when count data violate standard Poisson equi-dispersion ($\text{Var} > \text{Mean}$) and motivate mixture models (Negative Binomial).


## 4. Classwork 0: Predict Before Running
> ✍ **WRITE (Prediction)**:
> Suppose an opinion poll of $N = 20$ citizens yields $k = 13$ supporters of a policy.
> 1. Which value of the underlying support parameter $\theta \in [0, 1]$ will make the observed data most likely?
> 2. Will the likelihood function $\mathcal{L}(\theta \mid k=13)$ integrate to $1.0$ across $\theta \in [0, 1]$? Why or why not?


---
## 5. Classwork 1: Forward Generative Simulation
### A. Worked Example: Simulating Survey Samples from a Known Model
We begin in the **generative world**. Assume the true, ground-truth support for a policy is $\theta_0 = 0.65$. We simulate $S = 1,000$ independent survey organizations, each polling $N = 20$ voters.


In [ ]:
N_trials = 20
theta_true = 0.65
S_sims = 1000

# Forward simulation: drawing from Binomial(N=20, theta=0.65)
simulated_k = rng.binomial(n=N_trials, p=theta_true, size=S_sims)

print(f"Simulated {S_sims} survey replications:")
print(f"  Theoretical Mean:   {N_trials * theta_true:.2f}")
print(f"  Empirical Mean:     {simulated_k.mean():.2f}")
print(f"  Theoretical StdDev: {np.sqrt(N_trials * theta_true * (1 - theta_true)):.2f}")
print(f"  Empirical StdDev:   {simulated_k.std():.2f}")

# Histogram of simulated observations
counts, bins = np.histogram(simulated_k, bins=np.arange(-0.5, N_trials + 1.5, 1))
fig_sim = go.Figure(go.Bar(
    x=np.arange(0, N_trials + 1),
    y=counts / S_sims,
    marker_color='#2b6cb0',
    hovertemplate='<b>k = %{x}</b><br>Simulated Frequency: %{y:.3f}<extra></extra>'
))
fig_sim.update_layout(
    title=f'Forward Simulation: Binomial(N={N_trials}, θ={theta_true}) across {S_sims} Replications',
    xaxis_title='Simulated Successes k (out of 20)',
    yaxis_title='Relative Frequency',
    template='plotly_white',
    height=380
)
fig_sim.show()


### 💡 Suggested Experiments & Modifications for Forward Simulation
Try running these experiments in the sandbox cell below:

* **Experiment 1A (The Law of Large Numbers & Precision)**:
  * Increase sample size from $N = 20$ to $N = 200$ and $N = 2,000$ while keeping $\theta = 0.65$.
  * *What to watch*: Notice how the distribution of the sample proportion $\hat{p} = k/N$ becomes dramatically narrower. The standard error shrinks as $1/\sqrt{N}$.
* **Experiment 1B (Cognitive Science: Attentional Lapses & Rare Errors)**:
  * In sustained attention tasks (e.g. SART), participants make rare commission errors with probability $\theta = 0.04$ across $N = 50$ trials.
  * Simulate 1,000 participants. What percentage of participants make *zero* errors?
* **Experiment 1C (The Null Baseline: Fair Coin)**:
  * Set $\theta = 0.50$ with $N = 20$. Compare the symmetry and spread to the $\theta = 0.65$ baseline.


In [ ]:
# 🧪 STUDENT SANDBOX: Experiment with Forward Simulation
# Modify the parameters below and run to test your hypotheses!

sandbox_N = 200        # Try: 20, 50, 200, 1000
sandbox_theta = 0.65   # Try: 0.04 (rare errors), 0.50 (fair), 0.65 (majority)
sandbox_sims = 1000

# Run simulation
sandbox_k = rng.binomial(n=sandbox_N, p=sandbox_theta, size=sandbox_sims)
sandbox_prop = sandbox_k / sandbox_N

print(f"Results for N={sandbox_N}, θ={sandbox_theta}:")
print(f"  Mean Proportion: {sandbox_prop.mean():.4f} (True: {sandbox_theta:.4f})")
print(f"  SD Proportion:   {sandbox_prop.std():.4f} (Theoretical SE: {np.sqrt(sandbox_theta*(1-sandbox_theta)/sandbox_N):.4f})")
print(f"  Min k: {sandbox_k.min()}, Max k: {sandbox_k.max()}")


### 🛑 STOP 1: Synchronization Point
* In forward simulation, we fix the parameter $\theta_0$ and let data $k$ vary.
* Even when the generative model is 100% constant, individual sample outcomes fluctuate. This is **pure aleatory variability** (sampling noise).


---
## 6. Classwork 2: Inverse Reasoning — Evaluating the Likelihood Function
### A. Worked Example: Finding the MLE and Comparing Hypotheses
Now we switch to the **inferential perspective**. In empirical science, we do not know $\theta$; we only have our recorded data!
* **Fixed Data**: $k_{\text{obs}} = 13$ successes out of $N = 20$ trials.
* **Goal**: Sweep all candidate models $\theta \in [0, 1]$, plot the likelihood curve $\mathcal{L}(\theta \mid k=13)$, find the Maximum Likelihood Estimate (MLE), and calculate the Likelihood Ratio comparing $H_1: \theta = 0.65$ vs $H_2: \theta = 0.50$.


In [ ]:
k_obs = 13
N_obs = 20
theta_grid = np.linspace(0.001, 0.999, 500)

# Likelihood function: L(theta | k=13, N=20) = binom.pmf(13, 20, theta)
likelihood = stats.binom.pmf(k_obs, N_obs, theta_grid)

# Find MLE
mle_idx = np.argmax(likelihood)
mle_theta = theta_grid[mle_idx]
mle_val = likelihood[mle_idx]

# Point evaluations for competing hypotheses
h1_val = 0.65
h2_val = 0.50
l_h1 = stats.binom.pmf(k_obs, N_obs, h1_val)
l_h2 = stats.binom.pmf(k_obs, N_obs, h2_val)
lr_h1_h2 = l_h1 / l_h2

# Plot likelihood curve
fig_lik = go.Figure()
fig_lik.add_trace(go.Scatter(
    x=theta_grid, y=likelihood,
    mode='lines',
    line=dict(color='#d97706', width=2.5),
    fill='tozeroy',
    fillcolor='rgba(217, 119, 6, 0.12)',
    name=f'Likelihood L(θ | k={k_obs})'
))

# Highlight H1 and H2
fig_lik.add_trace(go.Scatter(
    x=[h2_val, h1_val], y=[l_h2, l_h1],
    mode='markers+text',
    text=[f'H2: θ={h2_val}<br>(L={l_h2:.4f})', f'H1: θ={h1_val}<br>(L={l_h1:.4f})'],
    textposition=['bottom left', 'top right'],
    marker=dict(size=10, color=['#c53030', '#276749']),
    name='Hypotheses'
))

fig_lik.update_layout(
    title=f'Likelihood Function L(θ | k={k_obs}, N={N_obs}) [Likelihood Ratio H1/H2 = {lr_h1_h2:.2f}]',
    xaxis_title='Candidate Parameter θ (Success Rate)',
    yaxis_title='Likelihood Value L(θ)',
    template='plotly_white',
    height=420
)
fig_lik.show()

print(f"Maximum Likelihood Estimate (MLE): θ_hat = {mle_theta:.3f} (Sample proportion: {k_obs}/{N_obs} = {k_obs/N_obs:.3f})")
print(f"Likelihood at H1 (θ={h1_val}): {l_h1:.4f}")
print(f"Likelihood at H2 (θ={h2_val}): {l_h2:.4f}")
print(f"Likelihood Ratio (LR):        {lr_h1_h2:.3f} -> The data are {lr_h1_h2:.2f}x more compatible with H1 than H2.")


### 💡 Suggested Experiments & Modifications for Likelihood Evaluation
Test these scenarios in the sandbox cell below:

* **Experiment 2A (The Explosive Power of Sample Size on Evidence)**:
  * Suppose a larger study observed the exact same $65\%$ proportion in $N = 100$ people ($k_{\text{obs}} = 65$).
  * Re-calculate the likelihood curve and Likelihood Ratio between $\theta = 0.65$ and $\theta = 0.50$.
  * *What to watch*: Notice how the curve becomes much narrower, and the evidence explodes from $\text{LR} \approx 2.5$ to $\text{LR} > 300$!
* **Experiment 2B (Extreme Boundary Evidence: Zero Errors or Ceiling Effect)**:
  * Set $k_{\text{obs}} = 0$ out of $N = 20$ (e.g. a participant makes zero errors on a task).
  * Where does the likelihood peak? What does the shape look like near the boundary $\theta = 0$?
* **Experiment 2C (Cognitive Science: Aha! Insight vs. Analytic Strategy)**:
  * In a semantic insight experiment (such as the Moroshkina Lab Triads study!), suppose solving via "Aha! insight" yields a success rate of $\theta_{\text{insight}} = 0.80$, whereas "deliberate analytic search" yields $\theta_{\text{analytic}} = 0.40$.
  * If a participant solves $k = 7$ out of $N = 10$ items, calculate the Likelihood Ratio. Does the data favor insight or analytic search?


In [ ]:
# 🧪 STUDENT SANDBOX: Test Your Own Likelihood Scenarios
# Modify k_sandbox, N_sandbox, and hypotheses:

k_sandbox = 65       # Try: 65 (large sample), 0 (zero errors), 7 (insight trials)
N_sandbox = 100      # Try: 100, 20, 10
hypo_1 = 0.65        # Try: 0.65, 0.80
hypo_2 = 0.50        # Try: 0.50, 0.40

# Compute likelihoods
lik_curve = stats.binom.pmf(k_sandbox, N_sandbox, theta_grid)
l_h1_sb = stats.binom.pmf(k_sandbox, N_sandbox, hypo_1)
l_h2_sb = stats.binom.pmf(k_sandbox, N_sandbox, hypo_2)
lr_sb = l_h1_sb / l_h2_sb if l_h2_sb > 0 else np.inf

print(f"Sandbox Experiment (k={k_sandbox}/{N_sandbox}):")
print(f"  MLE:                  {theta_grid[np.argmax(lik_curve)]:.3f}")
print(f"  Likelihood at H1:     {l_h1_sb:.6e}")
print(f"  Likelihood at H2:     {l_h2_sb:.6e}")
print(f"  Likelihood Ratio H1/H2: {lr_sb:.2f}x")


### 🛑 STOP 2: Numerical Proof — Why the Likelihood is NOT a Density
Let us use numerical integration (`scipy.integrate.trapezoid`) to verify the area under the likelihood curve:


In [ ]:
area_lik = trapezoid(likelihood, theta_grid)
theoretical_area = 1.0 / (N_obs + 1)

print(f"Numerical Area under L(θ | k=13, N=20): {area_lik:.5f}")
print(f"Exact Theoretical Area 1/(N+1) = 1/21:    {theoretical_area:.5f}")
print(f"Is the likelihood a valid probability density? {'YES' if np.isclose(area_lik, 1.0) else 'NO (Area ≠ 1.0)'}!")


---
## 7. Classwork 3: Predictions — Distinguishing the Bayesian Quartet
### A. Worked Example: Prior Predictive vs. Posterior Predictive Replications
Now we introduce prior uncertainty $\theta \sim \operatorname{Beta}(2, 2)$ and condition on our observed data $k = 13$ to generate predictions for future surveys of size $N = 20$.


In [ ]:
# 1. Prior: Beta(2, 2)
prior_a, prior_b = 2, 2
theta_prior_draws = rng.beta(prior_a, prior_b, size=2000)

# Prior predictive: fake data generated from prior parameter draws
y_prior_pred = rng.binomial(n=N_obs, p=theta_prior_draws)

# 2. Posterior: Beta(2 + 13, 2 + 7) = Beta(15, 9)
post_a = prior_a + k_obs
post_b = prior_b + (N_obs - k_obs)
theta_post_draws = rng.beta(post_a, post_b, size=2000)

# Posterior predictive: future data generated from posterior parameter draws
y_post_pred = rng.binomial(n=N_obs, p=theta_post_draws)

fig_pred = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>Prior Predictive Distribution</b><br><span style="font-size:11px;color:#64748b">Predictions before seeing data (Diffused)</span>',
        '<b>Posterior Predictive Distribution</b><br><span style="font-size:11px;color:#64748b">Predictions after conditioning on k=13 (Calibrated)</span>'
    ]
)

fig_pred.add_trace(go.Histogram(x=y_prior_pred, histnorm='probability', marker_color='#94a3b8', name='Prior Pred'), row=1, col=1)
fig_pred.add_trace(go.Histogram(x=y_post_pred, histnorm='probability', marker_color='#2563eb', name='Posterior Pred'), row=1, col=2)
fig_pred.add_vline(x=k_obs, line_dash='dash', line_color='#dc2626', annotation_text='Observed k=13', row=1, col=2)

fig_pred.update_layout(template='plotly_white', height=420, showlegend=False)
fig_pred.update_xaxes(title_text='Simulated k (out of 20)', range=[-0.5, 20.5])
fig_pred.update_yaxes(title_text='Probability')
fig_pred.show()

print(f"Prior Predictive Mean:     {y_prior_pred.mean():.2f} (SD = {y_prior_pred.std():.2f})")
print(f"Posterior Predictive Mean: {y_post_pred.mean():.2f} (SD = {y_post_pred.std():.2f})")
print(f"Posterior Parameter Mean:  {theta_post_draws.mean():.4f} (SD = {theta_post_draws.std():.4f})")


### 💡 Suggested Experiments & Modifications for Predictions
Test these scenarios in the sandbox cell below:

* **Experiment 3A (Prior Sensitivity)**:
  * Change the prior from $\operatorname{Beta}(2, 2)$ to a skeptical prior $\operatorname{Beta}(2, 10)$ or a strong prior $\operatorname{Beta}(20, 20)$.
  * How much does the posterior predictive distribution shift away from the raw data $k = 13$?
* **Experiment 3B (Individual Observation vs. Sample Mean)**:
  * Predict the outcome of just *one* single future respondent ($N_{\text{new}} = 1$) vs a large future replication sample ($N_{\text{new}} = 500$).
  * Observe why predicting an individual unit retains high discrete variance, while predicting a large sample mean approaches parameter certainty.


In [ ]:
# 🧪 STUDENT SANDBOX: Prior Sensitivity & Predictive Scales
# Test different priors:

prior_a_sb = 2      # Try: 2 (mild), 20 (dogmatic), 2 (skeptical)
prior_b_sb = 10     # Try: 2 (mild), 20 (dogmatic), 10 (skeptical)
N_future = 20       # Try: 1 (single person), 20 (standard poll), 500 (large study)

post_a_sb = prior_a_sb + k_obs
post_b_sb = prior_b_sb + (N_obs - k_obs)

theta_sb = rng.beta(post_a_sb, post_b_sb, size=2000)
y_future = rng.binomial(n=N_future, p=theta_sb)

print(f"Sandbox Prior: Beta({prior_a_sb}, {prior_b_sb}) + Data ({k_obs}/{N_obs}):")
print(f"  Posterior Mode:       {(post_a_sb - 1)/(post_a_sb + post_b_sb - 2):.3f}")
print(f"  Predicted Future Mean: {y_future.mean():.2f} out of {N_future}")
print(f"  Predicted 90% Interval: [{np.percentile(y_future, 5):.0f}, {np.percentile(y_future, 95):.0f}]")


---
## 8. Exit Record
> ✍ **WRITE (Summary Reflection)**:
> 1. **Estimand**: What theoretical quantity were we estimating in Classwork 2?
> 2. **Likelihood vs Density**: In your own words, why is the likelihood not a probability distribution over $\theta$?
> 3. **Epistemic vs Aleatory**: Why does the posterior predictive distribution for future survey respondents have greater variance than the posterior distribution of $\theta$?


---
## 9. 🏠 Optional Homework: Diagnosing Poisson Overdispersion
### A. Worked Example: Equi-dispersion vs. Latent Heterogeneity
In a standard Poisson process, variance equals mean: $\mathbb{E}[Y] = \operatorname{Var}(Y) = \lambda$. But empirical cognitive and social counts almost always exhibit **overdispersion** ($\operatorname{Var} > \text{Mean}$).

Below, we simulate 1,000 count observations from:
1. **Standard Poisson**: All individuals share identical rate $\lambda = 4.0$.
2. **Negative Binomial Mixture**: Individuals have heterogeneous abilities $\lambda_i \sim \operatorname{Gamma}(\alpha=2, \beta=0.5)$.


In [ ]:
n_counts = 1000
mean_rate = 4.0

# 1. Standard Poisson
y_poisson = rng.poisson(lam=mean_rate, size=n_counts)

# 2. Overdispersed Negative Binomial
alpha_disp = 2.0
lambda_subject = rng.gamma(shape=alpha_disp, scale=mean_rate / alpha_disp, size=n_counts)
y_negbin = rng.poisson(lam=lambda_subject)

print(f"Poisson:           Mean = {y_poisson.mean():.2f}, Var = {y_poisson.var():.2f} (Var/Mean = {y_poisson.var()/y_poisson.mean():.2f})")
print(f"Negative Binomial: Mean = {y_negbin.mean():.2f}, Var = {y_negbin.var():.2f} (Var/Mean = {y_negbin.var()/y_negbin.mean():.2f})")

fig_disp = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'Standard Poisson (Var/Mean = {y_poisson.var()/y_poisson.mean():.2f})',
        f'Negative Binomial Mixture (Var/Mean = {y_negbin.var()/y_negbin.mean():.2f})'
    ]
)
fig_disp.add_trace(go.Histogram(x=y_poisson, marker_color='#38a169', name='Poisson'), row=1, col=1)
fig_disp.add_trace(go.Histogram(x=y_negbin, marker_color='#e53e3e', name='NegBinomial'), row=1, col=2)
fig_disp.update_layout(template='plotly_white', height=380, showlegend=False)
fig_disp.update_xaxes(title_text='Count of Events')
fig_disp.show()


### 💡 Suggested Homework Experiments
* **Experiment 4A (Extreme Latent Heterogeneity)**:
  * In the sandbox below, decrease `alpha_disp` to $0.5$. Notice how the distribution develops an extreme heavy tail with outlier event counts exceeding 25!
* **Experiment 4B (Varying Exposure / Observation Time)**:
  * Suppose call-center or cognitive trial durations differ across observations ($T_i \in [0.5, 2.0]$ hours). Model event rate per unit time $\lambda_i \times T_i$.


In [ ]:
# 🧪 STUDENT SANDBOX: Test Overdispersion and Heavy Tails
alpha_test = 0.5  # Try: 0.5 (extreme overdispersion), 5.0 (near Poisson), 20.0 (virtually Poisson)

lambdas = rng.gamma(shape=alpha_test, scale=mean_rate / alpha_test, size=1000)
y_heavy = rng.poisson(lam=lambdas)

print(f"Overdispersion with alpha={alpha_test}:")
print(f"  Mean:     {y_heavy.mean():.2f}")
print(f"  Variance: {y_heavy.var():.2f} (Dispersion Index: {y_heavy.var()/y_heavy.mean():.2f})")
print(f"  Max count observed: {y_heavy.max()}")


---
## 10. Reproducibility Footer
* Python runtime: Python 3.12
* Key dependencies: `numpy`, `scipy`, `plotly`, `pandas`
* Platform: Google Colab & Local Jupyter environments
